# Analyse sémantique du graphe appris

Lit un fichier `learned_graph_foldN.csv` exporté par le notebook d'analyse GIRT,
résout chaque identifiant numérique de compétence vers son intitulé réel via
`name_from_id`, puis produit : la liste des arêtes les plus fortes, un tableau
des meilleures cibles par compétence source, et un diagramme nœuds-liens.

In [ ]:
import os
import sys
import csv
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

# Detect the repository root and load name_from_id from datas_management/.
ROOT = Path.cwd()
while not (ROOT / "datas_management").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "datas_management"))
import name_from_id
name_from_id.init()
print(f"Loaded name_from_id from: {ROOT / 'datas_management'}")

ANALYSIS_DIR = ROOT / "models" / "girt" / "analysis"
FIGURES_DIR  = ANALYSIS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ---- Configuration (replaces the original command-line arguments) ----
CSV_PATH       = ANALYSIS_DIR / "learned_graph_fold2.csv"   # graph CSV exported by girt_analysis
TOP_EDGES      = 30    # strongest edges to print
TOP_PER_SOURCE = 3     # strongest targets per source skill
GRAPH_TOP      = 50    # strongest edges to display in the plot
OUTPUT_PATH    = FIGURES_DIR / f"semantic_{CSV_PATH.stem}.pdf"

## Chargement des arêtes

In [ ]:
def load_edges(csv_path):
    """Read the CSV and replace numeric source/target names with the real titles."""
    edges = []
    with open(csv_path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            src_id = int(row["source_id"])
            tgt_id = int(row["target_id"])
            weight = float(row["weight"])
            src_name = name_from_id.get_competence_title(f"competence_{src_id}")
            tgt_name = name_from_id.get_competence_title(f"competence_{tgt_id}")
            edges.append({
                "src_id":   src_id,
                "tgt_id":   tgt_id,
                "src_name": src_name,
                "tgt_name": tgt_name,
                "weight":   weight,
            })
    return edges

## Vues textuelles

In [ ]:
def _truncate(s, n):
    return s if len(s) <= n else s[: n - 1] + "…"


def print_top_edges(edges, top_n=30):
    """Print the top-N edges sorted by weight."""
    ordered = sorted(edges, key=lambda e: -e["weight"])[:top_n]

    print(f"\nTop {top_n} edges by weight")
    print("=" * 110)
    print(f"{'#':>3}  {'Source':<45}    {'Target':<45}  {'Weight':>8}")
    print("-" * 110)
    for i, e in enumerate(ordered, 1):
        src = _truncate(e["src_name"], 43)
        tgt = _truncate(e["tgt_name"], 43)
        print(f"{i:>3}. {src:<45} → {tgt:<45} {e['weight']:>8.4f}")


def print_top_per_source(edges, top_n_per_source=3, max_sources=None):
    """For each source skill, list its top-N targets by weight."""
    by_source = defaultdict(list)
    for e in edges:
        by_source[e["src_id"]].append(e)
    for k in by_source:
        by_source[k].sort(key=lambda e: -e["weight"])

    sources = sorted(by_source.keys())
    if max_sources is not None:
        sources = sources[:max_sources]

    print(f"\nTop-{top_n_per_source} neighbours per source skill"
          f"  ({len(sources)} sources)")
    print("=" * 110)
    for src_id in sources:
        src_name = by_source[src_id][0]["src_name"]
        parts = [
            f"{_truncate(e['tgt_name'], 30)} ({e['weight']:.3f})"
            for e in by_source[src_id][:top_n_per_source]
        ]
        print(f"  {_truncate(src_name, 40):>40s}  →  " + ", ".join(parts))

## Exécution

In [ ]:
edges = load_edges(CSV_PATH)
print(f"Loaded {len(edges)} edges from {CSV_PATH}")

n_errors = sum(1 for e in edges
               if e["src_name"].startswith("[ERROR") or e["tgt_name"].startswith("[ERROR"))
if n_errors > 0:
    print(f"Warning: {n_errors} edges have at least one unresolved name.")

print_top_edges(edges, top_n=TOP_EDGES)
print_top_per_source(edges, top_n_per_source=TOP_PER_SOURCE)